# LeetCode 456 — 132 Pattern bằng Python

Link đề: https://leetcode.com/problems/132-pattern/ · Độ khó: **Medium** · Chủ đề: Stack, Monotonic Stack, Array, Binary Search, Ordered Set

Skeleton LeetCode:

```python
class Solution:
    def find132pattern(self, nums: List[int]) -> bool:
        ...
```

Bài này là bài **kinh điển để học ngăn xếp đơn điệu (monotonic stack)**. Ý tưởng "duyệt từ phải sang trái và nhớ giá trị `nums[k]` tốt nhất" rất gọn nhưng khó nghĩ ra nếu chưa quen, nên notebook đi từ vét cạn `O(n^3)`, rồi cải tiến dần.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tìm **ba chỉ số** `i < j < k` sao cho `nums[i] < nums[k] < nums[j]` (giá trị "1" nhỏ nhất, "3" lớn nhất, "2" ở giữa, theo thứ tự xuất hiện **1, 3, 2**).
2. Nắm các công cụ Python: **dãy con** (subsequence) và `itertools.combinations`, **prefix minimum** (nhỏ nhất tiền tố), `float("inf")`, **ngăn xếp** bằng `list` (`append`/`pop`/`[-1]`), `reversed`, `bisect` và cây **Fenwick (BIT)** nhỏ.
3. Đi từ **ba vòng lặp `O(n^3)`** → **cố định `j`, dùng min tiền tố `O(n^2)`** → **ngăn xếp đơn điệu duyệt từ phải sang trái `O(n)`** → **ngăn xếp đơn điệu duyệt từ trái sang phải** → **cây Fenwick `O(n log n)`**.
4. Hiểu mấu chốt: với mỗi `j`, phần tử "1" tốt nhất là **nhỏ nhất ở bên trái**; còn phần tử "2" cần lớn nhất có thể nhưng **vẫn nhỏ hơn "3"**. Duyệt từ phải sang trái cho phép **nhớ giá trị "2" lớn nhất đã tìm được**.
5. Tránh các lỗi kinh điển: dùng min **toàn mảng** (thay vì min bên trái `j`), pop ngăn xếp bằng `<=` thay vì `<`, khởi tạo `vk = 0` khi dữ liệu có số âm, và chỉ xét ba phần tử **liền kề**.

## 2. Đề bài & ví dụ

Cho mảng `n` số nguyên `nums`. Một **mẫu 132** là dãy con gồm ba số `nums[i]`, `nums[j]`, `nums[k]` sao cho

- `i < j < k` (đúng thứ tự xuất hiện), và
- `nums[i] < nums[k] < nums[j]`.

Trả về `true` nếu `nums` chứa mẫu 132, ngược lại `false`.

(Gọi là "132" vì ba số theo thứ tự xuất hiện có **cỡ** là 1, 3, 2: số đầu nhỏ nhất, số giữa lớn nhất, số cuối ở giữa.)

**Ví dụ 1**

- Input: `nums = [1,2,3,4]`
- Output: `false`
- Giải thích: mảng tăng dần nên không có mẫu 132.

**Ví dụ 2**

- Input: `nums = [3,1,4,2]`
- Output: `true`
- Giải thích: dãy con `[1, 4, 2]` thoả `1 < 2 < 4`.

**Ví dụ 3**

- Input: `nums = [-1,3,2,0]`
- Output: `true`
- Giải thích: có ba mẫu: `[-1,3,2]`, `[-1,3,0]` và `[-1,2,0]`.

**Ràng buộc:**

- `n == nums.length`
- `1 <= n <= 2 * 10^5`
- `-10^9 <= nums[i] <= 10^9`

In [1]:
from typing import List
from itertools import combinations
from bisect import bisect_left
import random
import time

examples = [([1, 2, 3, 4], False), ([3, 1, 4, 2], True), ([-1, 3, 2, 0], True)]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 3, 4] -> kỳ vọng False
[3, 1, 4, 2] -> kỳ vọng True
[-1, 3, 2, 0] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1. Dãy con (subsequence) và `itertools.combinations`

Dãy con giữ **thứ tự** xuất hiện nhưng có thể bỏ qua phần tử. `combinations(nums, 3)` sinh mọi bộ ba phần tử **theo đúng thứ tự trong mảng** (tức là `i < j < k`). Đây chính là cách vét cạn ngắn gọn, và ta dùng nó làm **oracle độc lập** để kiểm tra các thuật toán nhanh.

In [2]:
nums = [3, 1, 4, 2]
for trio in combinations(nums, 3):
    a, b, c = trio
    print(trio, "mẫu 132?", a < c < b)

(3, 1, 4) mẫu 132? False
(3, 1, 2) mẫu 132? False
(3, 4, 2) mẫu 132? False
(1, 4, 2) mẫu 132? True


### 3.2. Prefix minimum (nhỏ nhất tiền tố) và `float("inf")`

`pre[j] = min(nums[0..j-1])` là giá trị nhỏ nhất **nằm bên trái** `j`. Với `j = 0` chưa có phần tử nào nên đặt `pre[0] = +vô cực` (không thể làm "1"). Python có `float("inf")` và `-float("inf")` so sánh được với mọi số nguyên.

In [3]:
nums = [3, 1, 4, 2]
INF = float("inf")
pre = []
cur = INF
for x in nums:
    pre.append(cur)       # min của các phần tử đứng TRƯỚC x
    cur = min(cur, x)
print(pre)                 # [inf, 3, 1, 1]
print(INF > 10 ** 18, -INF < -10 ** 18)      # True True

[inf, 3, 1, 1]
True True


### 3.3. Ngăn xếp (stack) bằng `list`

`stk.append(x)` đẩy vào đỉnh; `stk.pop()` lấy đỉnh ra; `stk[-1]` xem đỉnh. Ngăn xếp "vào sau ra trước". Trong bài này ta cần nó để giữ **các ứng viên cho vị trí "3"**.

In [4]:
stk = []
stk.append(5)
stk.append(3)
print(stk, stk[-1])        # [5, 3] 3
print(stk.pop(), stk)      # 3 [5]
print(bool(stk), bool([])) # True False  (list rỗng là False)

[5, 3] 3
3 [5]
True False


### 3.4. Ngăn xếp đơn điệu (monotonic stack)

Ngăn xếp **đơn điệu giảm** luôn giữ các phần tử giảm dần từ đáy lên đỉnh. Khi đẩy `x` mới, ta **pop mọi phần tử nhỏ hơn `x`** trước, rồi mới push `x`. Việc pop cho ta biết: "phần tử vừa bị pop **có một phần tử lớn hơn nó đứng bên trái** (tức là `x`, nếu duyệt từ phải sang trái)". Đây đúng là quan hệ "3 đứng trước 2 và lớn hơn 2".

In [5]:
def demo_monotonic(arr):
    stk = []
    for x in arr:
        popped = []
        while stk and stk[-1] < x:
            popped.append(stk.pop())
        stk.append(x)
        print(f"x={x}: pop {popped} -> stk={stk}")

demo_monotonic([2, 4, 1, 3, 5])

x=2: pop [] -> stk=[2]
x=4: pop [2] -> stk=[4]
x=1: pop [] -> stk=[4, 1]
x=3: pop [1] -> stk=[4, 3]
x=5: pop [3, 4] -> stk=[5]


### 3.5. Duyệt ngược với `reversed` và `bisect_left`

`for x in reversed(nums)` duyệt từ cuối về đầu mà không tạo bản sao (khác `nums[::-1]`). `bisect_left(sorted_list, v)` cho vị trí chèn `v` vào list đã sắp xếp (dùng để **nén toạ độ** ở Cách 5).

In [6]:
nums = [3, 1, 4, 2]
print(list(reversed(nums)), nums[::-1])             # [2, 4, 1, 3] [2, 4, 1, 3]
vals = sorted(set(nums))
print(vals, [bisect_left(vals, v) + 1 for v in nums])   # xếp hạng 1..m: [3, 1, 4, 2]

[2, 4, 1, 3] [2, 4, 1, 3]
[1, 2, 3, 4] [3, 1, 4, 2]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Với `nums = [3, 1, 4, 2]`, hãy xét từng chỉ số `j` làm "số 3" (đỉnh):

| `j` | `nums[j]` | min bên trái (`i`) | Phần tử phía phải `j` | Có `k` với `min < nums[k] < nums[j]`? |
|---|---|---|---|---|
| 0 | 3 | không có | 1, 4, 2 | không có "1" ⇒ không |
| 1 | 1 | 3 | 4, 2 | cần `3 < v < 1` ⇒ không |
| 2 | 4 | 1 | 2 | `1 < 2 < 4` ⇒ **có** |
| 3 | 2 | 1 | (không có) | không có `k` |

Nhận xét quan trọng: với mỗi `j`, chọn `nums[i]` là **min bên trái** là tốt nhất, vì "1" càng nhỏ thì điều kiện `nums[i] < nums[k]` càng dễ thoả. Vậy bài toán thu về: **với mỗi `j`, có phần tử `k > j` nào nằm trong khoảng mở `(pre[j], nums[j])` không?**

In [7]:
nums = [3, 1, 4, 2]
n = len(nums)
INF = float("inf")
cur = INF
for j in range(n):
    lo, hi = cur, nums[j]
    ks = [nums[k] for k in range(j + 1, n) if lo < nums[k] < hi]
    print(f"j={j}: khoảng ({lo}, {hi}) -> các k hợp lệ: {ks}")
    cur = min(cur, nums[j])

j=0: khoảng (inf, 3) -> các k hợp lệ: []
j=1: khoảng (3, 1) -> các k hợp lệ: []
j=2: khoảng (1, 4) -> các k hợp lệ: [2]
j=3: khoảng (1, 2) -> các k hợp lệ: []


## 5. Cách 1 — Brute Force / baseline: ba vòng lặp

**Ý tưởng:** thử mọi `(i, j, k)` với `i < j < k` và kiểm tra `nums[i] < nums[k] < nums[j]`.

In [8]:
def find132_brute(nums):
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] < nums[k] < nums[j]:
                    return True
    return False


for nums, expected in examples:
    print(nums, find132_brute(nums), "kỳ vọng", expected)

[1, 2, 3, 4] False kỳ vọng False
[3, 1, 4, 2] True kỳ vọng True
[-1, 3, 2, 0] True kỳ vọng True


**Độ phức tạp:** `Time = O(n^3)`, `Space = O(1)`. Với `n = 2·10^5` là hoàn toàn không khả thi (`~10^15` phép).

**Nút thắt:** với mỗi `j`, ta thử **mọi `i`** trong khi chỉ cần giá trị **nhỏ nhất** bên trái (mục 4). Hơn nữa ta còn lặp lại việc tìm `k` cho các `j` gần nhau. Cần bỏ vòng `i`, rồi bỏ cả vòng `k`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — bỏ vòng `i`.** Với `j` cố định, chọn `nums[i]` = min tiền tố `pre[j]`. Chỉ cần tồn tại `k > j` với `pre[j] < nums[k] < nums[j]`.

**Bước 2 — bỏ vòng `k` bằng cách đảo chiều duyệt.** Duyệt `x = nums[·]` **từ phải sang trái** và nhớ giá trị `vk` = **giá trị "2" lớn nhất** mà ta đã biết là hợp lệ, nghĩa là tồn tại phần tử **lớn hơn nó, đứng bên trái nó** (trong phần đã duyệt). Khi gặp `x < vk`, thì `x` chính là "1": ở trái của cả cặp (3, 2), và nhỏ hơn "2".

Lý do chọn `vk` **lớn nhất**: "1" phải nhỏ hơn "2"; "2" càng lớn thì càng dễ có "1" nhỏ hơn nó. Cho nên chỉ cần nhớ **duy nhất một số** `vk` thay vì cả tập các số "2".

**Làm sao duy trì `vk`?** Dùng ngăn xếp đơn điệu giảm (đáy lớn, đỉnh nhỏ) chứa các ứng viên "3". Khi gặp `x` mới:

1. Nếu `x < vk` → tìm thấy mẫu 132, dừng.
2. Ngược lại, pop mọi phần tử **nhỏ hơn `x`** khỏi ngăn xếp; những phần tử đó có `x` (lớn hơn, đứng trước) làm "3", nên chúng là "2" hợp lệ; `vk` cập nhật thành **phần tử lớn nhất vừa bị pop** (pop cuối cùng là lớn nhất, vì ngăn xếp giảm dần từ đáy nên pop đỉnh→đáy cho giá trị tăng dần).
3. Push `x`.

Mỗi phần tử push và pop tối đa một lần ⇒ `O(n)`.

In [9]:
# Thí nghiệm nhỏ: duyệt từ phải sang trái trên [3,1,4,2]
nums = [3, 1, 4, 2]
vk = float("-inf")
stk = []
for x in reversed(nums):
    if x < vk:
        print("x =", x, "< vk =", vk, "=> tìm thấy mẫu 132")
        break
    while stk and stk[-1] < x:
        vk = stk.pop()
    stk.append(x)
    print("x =", x, "stk =", stk, "vk =", vk)

x = 2 stk = [2] vk = -inf
x = 4 stk = [4] vk = 2
x = 1 < vk = 2 => tìm thấy mẫu 132


## 7. Các cách tiếp theo

### Cách 2 — Cố định `j`, dùng min tiền tố, vòng `k` bên phải (`O(n^2)`)

**Giải thích từng đoạn:**

1. Duy trì `mn` = min của `nums[0..j-1]` (min tiền tố).
2. Với mỗi `j >= 1`: nếu `mn < nums[j]` mới có khả năng (cần "1" nhỏ hơn "3"), duyệt `k > j` tìm `mn < nums[k] < nums[j]`.
3. Sau khi xét `j`, cập nhật `mn = min(mn, nums[j])`.

In [10]:
def find132_prefix_min(nums):
    n = len(nums)
    mn = nums[0]
    for j in range(1, n):
        if mn < nums[j]:
            for k in range(j + 1, n):
                if mn < nums[k] < nums[j]:
                    return True
        mn = min(mn, nums[j])
    return False


for nums, expected in examples:
    print(nums, find132_prefix_min(nums), "kỳ vọng", expected)

[1, 2, 3, 4] False kỳ vọng False
[3, 1, 4, 2] True kỳ vọng True
[-1, 3, 2, 0] True kỳ vọng True


In [11]:
# Code nộp LeetCode — Cách 2 (đúng nhưng O(n^2), chậm với n = 2*10^5)
class Solution2:
    def find132pattern(self, nums: List[int]) -> bool:
        n = len(nums)
        mn = nums[0]
        for j in range(1, n):
            if mn < nums[j]:
                for k in range(j + 1, n):
                    if mn < nums[k] < nums[j]:
                        return True
            mn = min(mn, nums[j])
        return False


print(Solution2().find132pattern([3, 1, 4, 2]), Solution2().find132pattern([1, 2, 3, 4]))

True False


**Độ phức tạp Cách 2:** `Time = O(n^2)`, `Space = O(1)`.

### Cách 3 — Ngăn xếp đơn điệu duyệt từ **phải sang trái** (cách chuẩn)

Đây là ý ở mục 6. Ghi nhớ ba vai: `x` là "1" ứng viên (đang duyệt), phần tử bị pop mới nhất là `vk` ("2"), phần tử dẫn tới pop là "3".

**Giải thích từng đoạn:**

1. `vk = -inf` (chưa có "2"), `stk = []`.
2. Với `x` từ cuối về đầu: nếu `x < vk` → `True`.
3. `while stk and stk[-1] < x: vk = stk.pop()` — cập nhật `vk` với phần tử bị `x` "chặn" (x là "3" của nó).
4. `stk.append(x)`.
5. Hết vòng lặp: `False`.

In [12]:
def find132_stack_right(nums):
    vk = float("-inf")
    stk = []
    for x in reversed(nums):
        if x < vk:
            return True
        while stk and stk[-1] < x:
            vk = stk.pop()
        stk.append(x)
    return False


for nums, expected in examples:
    print(nums, find132_stack_right(nums), "kỳ vọng", expected)

[1, 2, 3, 4] False kỳ vọng False
[3, 1, 4, 2] True kỳ vọng True
[-1, 3, 2, 0] True kỳ vọng True


In [13]:
# Code nộp LeetCode — Cách 3
class Solution3:
    def find132pattern(self, nums: List[int]) -> bool:
        vk = float("-inf")
        stk = []
        for x in reversed(nums):
            if x < vk:
                return True
            while stk and stk[-1] < x:
                vk = stk.pop()
            stk.append(x)
        return False


print(Solution3().find132pattern([-1, 3, 2, 0]), Solution3().find132pattern([5, 4, 3]))

True False


**Độ phức tạp Cách 3:** mỗi phần tử push một lần, pop tối đa một lần ⇒ `Time = O(n)`, `Space = O(n)` (ngăn xếp trong trường hợp xấu nhất khi mảng giảm dần).

### Cách 4 — Ngăn xếp đơn điệu duyệt từ **trái sang phải** + min tiền tố

**Ý tưởng khác:** cố định **`k`** đang duyệt (là "2"). Cần tìm `j < k` với `nums[j] > nums[k]` và `pre[j] < nums[k]` (tồn tại "1" bên trái `j`).

- Giữ ngăn xếp chỉ số `j` với `nums[j]` **giảm dần** từ đáy lên đỉnh.
- Khi tới `k` (giá trị `x`): pop mọi `j` có `nums[j] <= x` (vì `x` đứng sau, không nhỏ hơn, nên `k` **thay thế tốt hơn** cho các `j` đó: cùng làm "3" thì `k` đứng muộn hơn nên `pre` nhỏ hơn hoặc bằng).
- Sau pop, mọi phần tử trong ngăn xếp có giá trị `> x`. Trong số đó **đỉnh** có chỉ số lớn nhất ⇒ `pre` nhỏ nhất (min tiền tố không tăng khi `j` tăng). Vậy chỉ cần xét đỉnh: nếu `pre[đỉnh] < x` → `True`.
- Push `k`.

In [14]:
def find132_stack_left(nums):
    n = len(nums)
    INF = float("inf")
    pre = [INF] * n               # pre[j] = min(nums[:j])
    cur = INF
    for j, x in enumerate(nums):
        pre[j] = cur
        cur = min(cur, x)
    stk = []                      # chỉ số j, nums[j] giảm dần
    for k, x in enumerate(nums):
        while stk and nums[stk[-1]] <= x:
            stk.pop()
        if stk and pre[stk[-1]] < x:
            return True
        stk.append(k)
    return False


for nums, expected in examples:
    print(nums, find132_stack_left(nums), "kỳ vọng", expected)

[1, 2, 3, 4] False kỳ vọng False
[3, 1, 4, 2] True kỳ vọng True
[-1, 3, 2, 0] True kỳ vọng True


In [15]:
# Code nộp LeetCode — Cách 4
class Solution4:
    def find132pattern(self, nums: List[int]) -> bool:
        n = len(nums)
        pre = [float("inf")] * n
        cur = float("inf")
        for j, x in enumerate(nums):
            pre[j] = cur
            cur = min(cur, x)
        stk = []
        for k, x in enumerate(nums):
            while stk and nums[stk[-1]] <= x:
                stk.pop()
            if stk and pre[stk[-1]] < x:
                return True
            stk.append(k)
        return False


print(Solution4().find132pattern([3, 1, 4, 2]), Solution4().find132pattern([1, 0, 1, -4, -3]))

True False


**Độ phức tạp Cách 4:** `Time = O(n)`, `Space = O(n)` (mảng `pre` và ngăn xếp).

### Cách 5 — Cây Fenwick (BIT) + nén toạ độ: đếm "có số nào trong khoảng" `O(n log n)`

**Ý tưởng:** quay lại câu hỏi của mục 4: với mỗi `j`, **có phần tử phía phải `j` nằm trong khoảng mở `(pre[j], nums[j])` không?**. Đó là bài toán **đếm số phần tử trong khoảng** trên một tập thay đổi:

1. **Nén toạ độ:** đổi các giá trị thành hạng `1..m` (mục 3.5) để dùng làm chỉ số của cây.
2. Ban đầu nạp **toàn bộ** mảng vào cây (đếm số lần xuất hiện mỗi hạng).
3. Duyệt `j` từ trái sang phải: **gỡ `nums[j]`** khỏi cây (khi đó cây chỉ chứa chỉ số `> j`), rồi truy vấn số phần tử có hạng trong `(rank(pre[j]), rank(nums[j]))`. Có ít nhất một ⇒ `True`.

Cây Fenwick hỗ trợ `add(i, delta)` và `query(i)` = tổng đếm hạng `<= i` đều `O(log m)`.

In [16]:
def find132_bit(nums):
    vals = sorted(set(nums))
    m = len(vals)
    rank = {v: i + 1 for i, v in enumerate(vals)}      # hạng 1..m
    tree = [0] * (m + 1)

    def add(i, d):
        while i <= m:
            tree[i] += d
            i += i & -i

    def query(i):                                      # số phần tử có hạng <= i
        s = 0
        while i > 0:
            s += tree[i]
            i -= i & -i
        return s

    for x in nums:
        add(rank[x], 1)
    mn = float("inf")
    for j, x in enumerate(nums):
        add(rank[x], -1)                               # cây chỉ còn các chỉ số > j
        if mn < x:
            if query(rank[x] - 1) - query(rank[mn]) > 0:   # có hạng nằm trong (rank[mn], rank[x])
                return True
        mn = min(mn, x)
    return False


for nums, expected in examples:
    print(nums, find132_bit(nums), "kỳ vọng", expected)

[1, 2, 3, 4] False kỳ vọng False
[3, 1, 4, 2] True kỳ vọng True
[-1, 3, 2, 0] True kỳ vọng True


In [17]:
# Code nộp LeetCode — Cách 5
class Solution5:
    def find132pattern(self, nums: List[int]) -> bool:
        vals = sorted(set(nums))
        m = len(vals)
        rank = {v: i + 1 for i, v in enumerate(vals)}
        tree = [0] * (m + 1)

        def add(i, d):
            while i <= m:
                tree[i] += d
                i += i & -i

        def query(i):
            s = 0
            while i > 0:
                s += tree[i]
                i -= i & -i
            return s

        for x in nums:
            add(rank[x], 1)
        mn = float("inf")
        for x in nums:
            add(rank[x], -1)
            if mn < x and query(rank[x] - 1) - query(rank[mn]) > 0:
                return True
            mn = min(mn, x)
        return False


print(Solution5().find132pattern([-1, 3, 2, 0]), Solution5().find132pattern([1, 2, 3, 4]))

True False


**Độ phức tạp Cách 5:** sắp xếp + nén `O(n log n)`, mỗi `add`/`query` `O(log n)` ⇒ `Time = O(n log n)`, `Space = O(n)`. Chậm hơn ngăn xếp nhưng **tổng quát hơn**: đổi điều kiện khoảng là xong, không cần nghĩ lại thuật toán.

**Tóm tắt:** Cách 1/2 để hiểu bài; Cách 3 là bản nộp chuẩn; Cách 4 nhìn từ phía "2"; Cách 5 là công cụ tổng quát cho "tồn tại phần tử trong khoảng".

## 8. TRACE từng bước

`trace_right` in từng bước của Cách 3, `trace_left` in từng bước của Cách 4.

Ví dụ `[3, 1, 4, 2]` (kết quả `True`) và `[1, 2, 3, 4]` (kết quả `False`).

In [18]:
def trace_right(nums):
    vk = float("-inf")
    stk = []
    print(f"{'x':>4} | {'so x<vk?':^9} | {'pop':^12} | {'stk sau':^16} | {'vk':>5}")
    for x in reversed(nums):
        if x < vk:
            print(f"{x:>4} | {'CÓ -> True':^9} | {'':^12} | {str(stk):^16} | {vk:>5}")
            return True
        popped = []
        while stk and stk[-1] < x:
            vk = stk.pop()
            popped.append(vk)
        stk.append(x)
        print(f"{x:>4} | {'không':^9} | {str(popped):^12} | {str(stk):^16} | {vk:>5}")
    return False


print(trace_right([3, 1, 4, 2]))
print()
print(trace_right([1, 2, 3, 4]))

   x | so x<vk?  |     pop      |     stk sau      |    vk
   2 |   không   |      []      |       [2]        |  -inf
   4 |   không   |     [2]      |       [4]        |     2
   1 | CÓ -> True |              |       [4]        |     2
True

   x | so x<vk?  |     pop      |     stk sau      |    vk
   4 |   không   |      []      |       [4]        |  -inf
   3 |   không   |      []      |      [4, 3]      |  -inf
   2 |   không   |      []      |    [4, 3, 2]     |  -inf
   1 |   không   |      []      |   [4, 3, 2, 1]   |  -inf
False


Đọc bảng `[3, 1, 4, 2]` (duyệt `2, 4, 1, 3`):

| Bước | `x` | Diễn giải |
|---|---|---|
| 1 | 2 | ngăn xếp rỗng, push; `vk = -inf` |
| 2 | 4 | `2 < 4` nên pop `2` ⇒ `vk = 2` (có "3" = 4 đứng trước "2" = 2) |
| 3 | 1 | `1 < vk = 2` ⇒ **tìm thấy**: `1 < 2 < 4`, ứng với `nums = [1, 4, 2]` |

In [19]:
def trace_left(nums):
    n = len(nums)
    INF = float("inf")
    pre = [INF] * n
    cur = INF
    for j, x in enumerate(nums):
        pre[j] = cur
        cur = min(cur, x)
    print("pre =", pre)
    stk = []
    for k, x in enumerate(nums):
        popped = []
        while stk and nums[stk[-1]] <= x:
            popped.append(stk.pop())
        top = stk[-1] if stk else None
        ok = top is not None and pre[top] < x
        print(f"k={k} x={x:>2} pop idx {popped} | đỉnh idx={top} | pre[đỉnh]<x ? {ok}")
        if ok:
            return True
        stk.append(k)
    return False


print(trace_left([3, 1, 4, 2]))
print(trace_left([-1, 3, 2, 0]))

pre = [inf, 3, 1, 1]
k=0 x= 3 pop idx [] | đỉnh idx=None | pre[đỉnh]<x ? False
k=1 x= 1 pop idx [] | đỉnh idx=0 | pre[đỉnh]<x ? False
k=2 x= 4 pop idx [1, 0] | đỉnh idx=None | pre[đỉnh]<x ? False
k=3 x= 2 pop idx [] | đỉnh idx=2 | pre[đỉnh]<x ? True
True
pre = [inf, -1, -1, -1]
k=0 x=-1 pop idx [] | đỉnh idx=None | pre[đỉnh]<x ? False
k=1 x= 3 pop idx [0] | đỉnh idx=None | pre[đỉnh]<x ? False
k=2 x= 2 pop idx [] | đỉnh idx=1 | pre[đỉnh]<x ? True
True


Với `[-1, 3, 2, 0]`: tại `k = 2` (`x = 2`) đỉnh ngăn xếp là chỉ số `1` (giá trị `3 > 2`), `pre[1] = -1 < 2` ⇒ `True`, tương ứng mẫu `-1, 3, 2`.

In [20]:
# Trace một mảng "khó": có nhiều lần pop liên tiếp trước khi tìm thấy
trace_right([6, 12, 3, 4, 6, 11, 20])

   x | so x<vk?  |     pop      |     stk sau      |    vk
  20 |   không   |      []      |       [20]       |  -inf
  11 |   không   |      []      |     [20, 11]     |  -inf
   6 |   không   |      []      |   [20, 11, 6]    |  -inf
   4 |   không   |      []      |  [20, 11, 6, 4]  |  -inf
   3 |   không   |      []      | [20, 11, 6, 4, 3] |  -inf
  12 |   không   | [3, 4, 6, 11] |     [20, 12]     |    11
   6 | CÓ -> True |              |     [20, 12]     |    11


True

## 9. Code nộp LeetCode

Chọn Cách 3 (ngăn xếp phải→trái): ngắn, `O(n)` thời gian, ít lỗi biên nhất.

In [21]:
class Solution:
    def find132pattern(self, nums: List[int]) -> bool:
        vk = float("-inf")
        stk = []
        for x in reversed(nums):
            if x < vk:
                return True
            while stk and stk[-1] < x:
                vk = stk.pop()
            stk.append(x)
        return False


for nums, expected in examples:
    assert Solution().find132pattern(nums) == expected
print("OK các ví dụ đề")

OK các ví dụ đề


## 10. Vì sao thuật toán đúng

Gọi `R` là phần đã duyệt (các chỉ số **lớn hơn** chỉ số của `x` hiện tại, vì ta duyệt từ phải sang trái).

**Định nghĩa.** Gọi một giá trị `v = nums[k]` (`k` thuộc `R`) là **"2 hợp lệ"** nếu trong `R` có `j < k` với `nums[j] > v`. Khẳng định: sau khi xử lý xong mỗi phần tử, `vk` = **giá trị lớn nhất trong các "2 hợp lệ"** (hoặc `-inf` nếu chưa có).

**Bất biến 1 — ngăn xếp.** Ta pop mọi phần tử `< x` rồi mới push `x`, nên ngăn xếp luôn **không tăng từ đáy lên đỉnh** (đỉnh nhỏ nhất). Mọi phần tử đã bị pop, hoặc đang nằm trong ngăn xếp, đều thuộc `R`.

**Bất biến 2 — mọi phần tử bị pop là "2 hợp lệ".** Phần tử `v` chỉ bị pop khi gặp `x > v`, mà `x` đứng bên trái `v` ⇒ `(j, k)` = (vị trí `x`, vị trí `v`) thoả định nghĩa. Vậy `vk` (một phần tử bị pop) luôn là một "2 hợp lệ".

**Bất biến 3 — không bỏ sót "2 hợp lệ".** Giả sử `v = nums[k]` là "2 hợp lệ" với `j < k`, `nums[j] > v`. Khi ta xử lý `nums[j]`: nếu `v` còn trong ngăn xếp thì nó nằm ở trên hoặc ngang các phần tử bị pop (vì `v < nums[j]` và ngăn xếp không tăng từ đáy lên, mọi phần tử từ đỉnh xuống đến `v` đều `<= v < nums[j]` nên **đều bị pop** trước khi dừng), do đó `v` bị pop; nếu `v` đã bị pop sớm hơn thì đã được tính. Vậy mọi "2 hợp lệ" đều từng bị pop.

**Bất biến 4 — `vk` là lớn nhất (và không giảm).** Trong **một** lần xử lý `x`, các phần tử bị pop theo thứ tự từ đỉnh xuống nên giá trị **tăng dần**; phép gán `vk = pop()` ở mỗi lần pop khiến `vk` cuối cùng bằng phần tử pop **sau cùng**, tức lớn nhất trong lần đó. Giữa hai bước liên tiếp: sau bước của `x`, đỉnh ngăn xếp chính là `x`, và `x` lớn hơn mọi giá trị vừa bị pop ở bước đó (vì chỉ pop những phần tử `< x`). Ở bước kế tiếp, nếu có pop thì phần tử bị pop **đầu tiên** chính là `x` cũ, nên `vk` mới `>= x_cũ >` `vk` cũ. Vậy `vk` không bao giờ giảm; kết hợp với Bất biến 2–3, `vk` luôn bằng "2 hợp lệ" lớn nhất.

**Tính đúng của kết luận.**

- Nếu `x < vk`: có `k, j` trong `R` với `nums[k] = vk`, `nums[j] > vk`, `j < k`, và `x` đứng trước `j` ⇒ `(i, j, k)` với `nums[i] = x < nums[k] = vk < nums[j]` là mẫu 132 ⇒ `True` đúng.
- Nếu có mẫu 132 `(i, j, k)`: khi duyệt tới `i`, cặp `(j, k)` đã nằm trong `R`, nên `nums[k]` là "2 hợp lệ" và `vk >= nums[k] > nums[i]` ⇒ ta trả `True` (có thể sớm hơn). Nếu không có mẫu, điều kiện `x < vk` không bao giờ đúng ⇒ trả `False`.

Mục 12 còn kiểm chứng thực nghiệm bằng vét cạn `combinations` trên hàng nghìn mảng nhỏ và toàn bộ hoán vị của `[1..6]`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Ba vòng lặp | `O(n^3)` | `O(1)` | chỉ dùng để kiểm chứng |
| 2. Min tiền tố + vòng `k` | `O(n^2)` | `O(1)` | TLE với `n = 2·10^5` |
| 3. Ngăn xếp phải→trái | `O(n)` | `O(n)` | **nộp bài** |
| 4. Ngăn xếp trái→phải | `O(n)` | `O(n)` | cần mảng `pre` |
| 5. Fenwick | `O(n log n)` | `O(n)` | tổng quát, nhưng hằng số lớn |

Cách 3: mỗi phần tử vào ngăn xếp đúng 1 lần và ra nhiều nhất 1 lần ⇒ tổng số thao tác ngăn xếp `≤ 2n`, dù có `while` lồng bên trong `for` (phân tích **khấu hao**).

## 12. Bộ test

### 12.1. Ví dụ đề + edge case

Đối chiếu **mọi cách** với nhau và với **oracle độc lập** `combinations`.

In [22]:
def oracle(nums):
    """Vét cạn độc lập: mọi bộ ba theo thứ tự xuất hiện."""
    return any(a < c < b for a, b, c in combinations(nums, 3))


solutions = {
    "brute": find132_brute,
    "prefix_min": find132_prefix_min,
    "stack_right": find132_stack_right,
    "stack_left": find132_stack_left,
    "bit": find132_bit,
    "Solution": lambda nums: Solution().find132pattern(list(nums)),
}

edge = [
    ([1, 2, 3, 4], False),
    ([3, 1, 4, 2], True),
    ([-1, 3, 2, 0], True),
    ([1], False),                    # n = 1
    ([1, 2], False),                 # n = 2
    ([1, 3, 2], True),               # tối thiểu 3 phần tử
    ([2, 3, 1], False),              # 2 3 1 là mẫu 231
    ([3, 2, 1], False),              # giảm dần
    ([1, 1, 1], False),              # bằng nhau
    ([1, 2, 2], False),              # nums[k] == nums[j]
    ([2, 2, 1, 3], False),
    ([-5, -3, -4], True),            # toàn số âm
    ([10 ** 9, -10 ** 9, 10 ** 9 - 1], False),
    ([-10 ** 9, 10 ** 9, 0], True),
    ([1, 0, 1, -4, -3], False),      # ví dụ nổi tiếng gây sai
    ([3, 5, 0, 3, 4], True),
]
for nums, expected in edge:
    assert oracle(nums) == expected, ("oracle", nums)
    for name, fn in solutions.items():
        assert fn(nums) == expected, (name, nums, expected)
print("OK", len(edge), "ca, mọi cách đều khớp oracle")

OK 16 ca, mọi cách đều khớp oracle


### 12.2. Phản ví dụ cho các cách làm sai

Ba "lời giải nghe có lý" nhưng sai: (1) dùng min toàn mảng, (2) chỉ xét ba phần tử liền kề, (3) pop bằng `<=`.

In [23]:
def wrong_global_min(nums):
    mn = min(nums)                       # SAI: phải là min bên TRÁI của j
    for j in range(len(nums)):
        for k in range(j + 1, len(nums)):
            if mn < nums[k] < nums[j]:
                return True
    return False

def wrong_adjacent(nums):
    return any(nums[i] < nums[i + 2] < nums[i + 1] for i in range(len(nums) - 2))

def wrong_pop_le(nums):
    vk = float("-inf")
    stk = []
    for x in reversed(nums):
        if x < vk:
            return True
        while stk and stk[-1] <= x:      # SAI: <= thay vì <
            vk = stk.pop()
        stk.append(x)
    return False

tests = {
    "min toàn mảng": (wrong_global_min, [5, 4, 1]),
    "ba phần tử liền kề": (wrong_adjacent, [1, 4, 0, 2]),
    "pop bằng <=": (wrong_pop_le, [1, 2, 2]),
}
for name, (fn, nums) in tests.items():
    print(f"{name:20s} {nums} -> sai: {fn(nums)}, đúng: {oracle(nums)}")
    assert fn(nums) != oracle(nums)

min toàn mảng        [5, 4, 1] -> sai: True, đúng: False
ba phần tử liền kề   [1, 4, 0, 2] -> sai: False, đúng: True
pop bằng <=          [1, 2, 2] -> sai: True, đúng: False


### 12.3. Đối chiếu ngẫu nhiên có seed (mảng nhỏ) với oracle

Sinh mảng ngẫu nhiên `n <= 9`, giá trị nhỏ (để có nhiều giá trị bằng nhau) và giá trị âm.

In [24]:
random.seed(456)
true_count = 0
for _ in range(3000):
    n = random.randint(1, 9)
    lo = random.choice([-3, 0, -10])
    hi = lo + random.choice([3, 6, 12])
    nums = [random.randint(lo, hi) for _ in range(n)]
    exp = oracle(nums)
    true_count += exp
    for name, fn in solutions.items():
        assert fn(nums) == exp, (name, nums, exp)
print("OK 3000 mảng; số mảng có mẫu 132:", true_count)

OK 3000 mảng; số mảng có mẫu 132: 1304


### 12.4. Hoán vị và mảng lớn

Mọi hoán vị của `[1..6]` (720 mảng) đối chiếu với oracle, rồi mảng lớn `n = 2·10^5` so sánh các cách `O(n)` / `O(n log n)`.

In [25]:
from itertools import permutations

cnt = 0
for p in permutations(range(1, 7)):
    exp = oracle(p)
    assert find132_stack_right(p) == find132_stack_left(p) == find132_bit(p) == exp, p
    cnt += exp
print("hoán vị: có mẫu 132 ở", cnt, "/ 720 (còn lại 132 hoán vị tránh 132 = số Catalan C6 = 132)")
assert 720 - cnt == 132

hoán vị: có mẫu 132 ở

 588 / 720 (còn lại 132 hoán vị tránh 132 = số Catalan C6 = 132)


In [26]:
random.seed(7)
n = 200000
fast = {"stack_right": find132_stack_right, "stack_left": find132_stack_left, "bit": find132_bit}

arrays = {
    "tăng dần (False)": list(range(n)),
    "giảm dần (False)": list(range(n, 0, -1)),
    "ngẫu nhiên": [random.randint(-10 ** 9, 10 ** 9) for _ in range(n)],
    "tăng rồi cuối có mẫu": list(range(n - 3)) + [10 ** 9, -1, 5] ,
    "răng cưa": [(-1) ** i * i for i in range(n)],
}
for name, arr in arrays.items():
    res = {}
    for fname, fn in fast.items():
        t0 = time.perf_counter()
        res[fname] = fn(arr)
        res[fname + "_t"] = round(time.perf_counter() - t0, 3)
    assert res["stack_right"] == res["stack_left"] == res["bit"], name
    print(f"{name:24s} -> {res['stack_right']}  (thời gian: {res['stack_right_t']}s / {res['stack_left_t']}s / {res['bit_t']}s)")

tăng dần (False)         -> False  (thời gian: 0.011s / 0.024s / 0.5s)


giảm dần (False)         -> False  (thời gian: 0.014s / 0.029s / 0.275s)


ngẫu nhiên               -> True  (thời gian: 0.0s / 0.008s / 0.314s)


tăng rồi cuối có mẫu     -> True  (thời gian: 0.011s / 0.027s / 0.177s)


răng cưa                 -> False  (thời gian: 0.014s / 0.029s / 0.475s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng min **toàn mảng** thay vì min **bên trái** `j`

Nhớ rằng `i < j`; số nhỏ nhất của cả mảng có thể nằm bên phải `j`.

In [27]:
nums = [5, 4, 1]
print("sai :", wrong_global_min(nums))      # True (vì min toàn mảng = 1)
print("đúng:", find132_stack_right(nums))   # False (1 nằm ở cuối, không đứng trước)

sai : True
đúng: False


**Đúng:** dùng `pre[j]` = `min(nums[:j])` (Cách 2/4), hoặc duyệt phải→trái để `x` chính là "1" đứng **trước** cặp (Cách 3).

### Lỗi 2: Chỉ kiểm tra ba phần tử liền kề

Dãy con **không cần liền kề** (`i < j < k` chứ không phải `j = i + 1`, `k = j + 1`).

In [28]:
nums = [1, 4, 0, 2]
print("sai (liền kề):", wrong_adjacent(nums))    # False
print("đúng         :", find132_stack_right(nums), oracle(nums))   # True: 1, 4, 2

sai (liền kề): False
đúng         : True True


**Đúng:** dùng `combinations`/vòng lặp độc lập chỉ số, hoặc thuật toán chuẩn.

### Lỗi 3: Pop bằng `<=` thay vì `<` (hoặc xét bằng nhau)

Đề yêu cầu **nghiêm ngặt**: `nums[i] < nums[k] < nums[j]`. Hai số bằng nhau không tạo được "3" và "2".

In [29]:
nums = [1, 2, 2]
print("sai (<=):", wrong_pop_le(nums))             # True
print("đúng    :", find132_stack_right(nums))      # False

sai (<=): True
đúng    : False


**Đúng:** `while stk and stk[-1] < x`.

### Lỗi 4: Khởi tạo `vk = 0` khi có số âm

Nếu `vk = 0` thì phần tử đầu tiên âm (`x < 0`) đã bị coi là "1" dù chưa có cặp (3, 2).

In [30]:
def wrong_vk_zero(nums):
    vk = 0                                # SAI: phải là -inf
    stk = []
    for x in reversed(nums):
        if x < vk:
            return True
        while stk and stk[-1] < x:
            vk = stk.pop()
        stk.append(x)
    return False

nums = [-5, -4, -3]
print("sai :", wrong_vk_zero(nums))         # True
print("đúng:", find132_stack_right(nums))   # False

sai : True
đúng: False


**Đúng:** `vk = float("-inf")` (hoặc `-inf` bằng `-10**10`, nhỏ hơn mọi ràng buộc).

### Lỗi 5: Hai vòng lặp lồng nhau cho `n = 2·10^5`

Cách 2 đúng nhưng `O(n^2)`. Mảng xấu cho nó là mảng **không có mẫu** mà nhánh `mn < nums[j]` luôn đúng, ví dụ **tăng dần**: với mỗi `j` phải quét hết phần bên phải.

In [31]:
n = 6000
inc = list(range(n))
t0 = time.perf_counter(); r2 = find132_prefix_min(inc); t2 = time.perf_counter() - t0
t0 = time.perf_counter(); r3 = find132_stack_right(inc); t3 = time.perf_counter() - t0
print(r2, r3, f"Cách 2: {t2:.4f}s, Cách 3: {t3:.6f}s")
assert r2 == r3 == False

False False Cách 2: 0.7901s, Cách 3: 0.000356s


Với `n = 6000` đã chậm hơn hàng nghìn lần; với `n = 2·10^5` thì Cách 2 chạy hàng phút. **Đúng:** dùng ngăn xếp (Cách 3) hoặc Fenwick (Cách 5).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy **"tồn tại bộ ba theo thứ tự với quan hệ lớn/nhỏ"** → nghĩ: cố định phần tử **giữa** (`j`), rồi tính "tốt nhất bên trái" và "tốt nhất bên phải".
- Cặp **"a đứng trước b và a > b"** (phần tử lớn hơn đứng trước) → **ngăn xếp đơn điệu**: pop chính là lúc phát hiện cặp đó.
- Khi cần "giá trị lớn nhất nhỏ hơn X ở bên phải" → duyệt phải→trái và **nhớ một giá trị** (`vk`) thường đủ; nếu bài đổi sang đếm hay cần khoảng tổng quát → **Fenwick/Segment Tree/ordered set**.
- Mẹo nhớ: `x < vk` ("1 nhỏ hơn 2") và `stk[-1] < x` ("pop 2 khi gặp 3").
- Cùng họ: 496 (Next Greater Element), 739 (Daily Temperatures), 84 (Largest Rectangle), 334 (Increasing Triplet), 1762, 2073.

## 15. Biến thể của bài

1. **Mẫu 123** (`nums[i] < nums[j] < nums[k]`): LeetCode 334 — `O(n)` với hai biến `first`, `second`.
2. **Trả về chỉ số** `(i, j, k)` của một mẫu 132 (bài thực tế hơn).
3. **Đếm số bộ ba** 132: `O(n log n)` với Fenwick (hoặc `O(n^2)`/`O(n^3)` nếu nhỏ).
4. **Cho phép bằng nhau** (không nghiêm ngặt: `nums[i] <= nums[k] <= nums[j]`): đổi điều kiện `<` thành `<=` ở ngăn xếp.

Dưới đây là biến thể 1, 2 và 3 (có test).

In [32]:
def has_123(nums):
    """Mẫu 123: i<j<k, nums[i]<nums[j]<nums[k]."""
    first = second = float("inf")
    for x in nums:
        if x <= first:
            first = x
        elif x <= second:
            second = x
        else:
            return True
    return False

def find132_indices(nums):
    """Trả (i, j, k) hợp lệ hoặc None (dùng ngăn xếp trái->phải, lưu chỉ số)."""
    n = len(nums)
    pre_idx = [-1] * n                    # chỉ số của min bên trái j
    best = -1
    for j in range(n):
        pre_idx[j] = best
        if best == -1 or nums[j] < nums[best]:
            best = j
    stk = []
    for k, x in enumerate(nums):
        while stk and nums[stk[-1]] <= x:
            stk.pop()
        if stk:
            j = stk[-1]
            i = pre_idx[j]
            if i != -1 and nums[i] < x:
                return (i, j, k)
        stk.append(k)
    return None

def count_132(nums):
    """Đếm bộ ba (i,j,k) — vét cạn O(n^3) cho mảng nhỏ."""
    n = len(nums)
    return sum(1 for i in range(n) for j in range(i + 1, n) for k in range(j + 1, n) if nums[i] < nums[k] < nums[j])

print(has_123([1, 2, 3]), has_123([5, 4, 3]), has_123([2, 1, 5, 0, 4, 6]))
print(find132_indices([3, 1, 4, 2]), find132_indices([1, 2, 3, 4]))
print(count_132([-1, 3, 2, 0]))     # 3 (khớp ví dụ 3 của đề)

random.seed(15)
for _ in range(400):
    nums = [random.randint(-4, 4) for _ in range(random.randint(1, 8))]
    assert has_123(nums) == any(a < b < c for a, b, c in combinations(nums, 3))
    r = find132_indices(nums)
    assert (r is not None) == oracle(nums)
    if r:
        i, j, k = r
        assert i < j < k and nums[i] < nums[k] < nums[j]
print("OK biến thể")

True False True
(1, 2, 3) None
3
OK biến thể


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Ba vòng lặp | thử mọi `(i, j, k)` | `O(n^3)` | `O(1)` | chỉ để kiểm chứng, `n` rất nhỏ |
| 2. Min tiền tố + quét `k` | cố định `j`, `i` = min trái | `O(n^2)` | `O(1)` | `n` vài nghìn, cần code đơn giản |
| 3. Ngăn xếp phải→trái | nhớ "2" lớn nhất `vk` | `O(n)` | `O(n)` | **nộp bài**, nên thuộc |
| 4. Ngăn xếp trái→phải | cố định "2", xét đỉnh + `pre` | `O(n)` | `O(n)` | khi muốn trả chỉ số, hoặc dữ liệu đến từ trái |
| 5. Fenwick | đếm phần tử trong `(pre, nums[j])` | `O(n log n)` | `O(n)` | điều kiện khoảng phức tạp / cần đếm số bộ ba |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `Solution().find132pattern([2, 4, 3])`, `Solution().find132pattern([2, 3, 4])`, `Solution().find132pattern([4, 2, 3])`, `Solution().find132pattern([1, 3, 2, 2])`. Điền `ans1` là list bốn giá trị `True`/`False` (hiện `None`).

**Bài 2 (tính tay).** Chạy bảng ngăn xếp phải→trái (`x`, pop, `stk`, `vk`) cho `nums = [6, 12, 3, 4, 6, 11, 20]`. Cho biết ở bước nào (đếm từ 1) thì dừng, `vk` lúc đó bằng bao nhiêu, và mẫu 132 nào ứng với nó. Điền `ans2_step` (số thứ tự `x` khi dừng, tính từ 1) và `ans2_triple` (bộ ba giá trị theo thứ tự xuất hiện trong mảng).

**Bài 3 (điền TODO).** Hoàn thành `find132_todo(nums)` theo Cách 3.

In [33]:
ans1 = None
ans2_step = None
ans2_triple = None

def find132_todo(nums):
    # TODO 1: vk = float("-inf"); stk = []
    # TODO 2: for x in reversed(nums):
    # TODO 3:     nếu x < vk: return True
    # TODO 4:     while stk and stk[-1] < x: vk = stk.pop()
    # TODO 5:     stk.append(x)
    # TODO 6: return False
    pass

# KHÔNG gọi find132_todo ở đây để Run All không lỗi.

**Bài 4 (tự viết lại).** Viết `find132_oracle(nums)` **chỉ bằng `itertools.combinations`** và `any(...)`, một dòng, không tham khảo mục 12.

**Bài 5 (biến thể / mini challenge).** (a) Viết `has_231(nums)`: tồn tại `i < j < k` với `nums[k] < nums[i] < nums[j]` (mẫu "231"). Gợi ý: đảo vai trò, thử duyệt trái→phải với ngăn xếp giảm và nhớ "giá trị lớn nhất đã bị pop" (đối xứng với bài gốc). (b) Viết `count_132_fast(nums)` đếm số bộ ba bằng `O(n^2)` (với mỗi `j`, đếm số cặp `(i, k)`); so với `count_132`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: kiểm tra từng mảng có đúng dạng "nhỏ, lớn, ở giữa". `[2,4,3]` có `2 < 3 < 4`. Bài 3: bám sát mục 9.
- **Mức 2 (vừa):** Bài 2: duyệt `20, 11, 6, 4, 3, 12, 6`; để ý lúc `12` pop liên tiếp bốn phần tử và `vk` nhảy lên bao nhiêu. Bài 4: `any(a < c < b for a, b, c in combinations(nums, 3))`.
- **Mức 3 (sâu):** Bài 5(a): mẫu 231 = `nums[k] < nums[i] < nums[j]`. Cách nghĩ: duyệt trái→phải, ngăn xếp giảm; nhớ `vi` là "2" lớn nhất đã bị pop (cặp `(i, j)` có `nums[i] < nums[j]`); khi gặp `x < vi` thì có mẫu. Bài 5(b): với mỗi `j`, đếm `cnt_i` ứng với mọi `k > j`: gộp theo `k`: số `i < j` có `nums[i] < nums[k]`; có thể duyệt hai mảng phụ hoặc dùng vòng `O(n^2)` bằng cách duyệt cặp `(j, k)` và đếm `i` bằng prefix đếm.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [34]:
# Bài 1: [2,4,3] True; [2,3,4] False; [4,2,3] False; [1,3,2,2] True
ans1 = [True, False, False, True]
assert ans1 == [Solution().find132pattern(a) for a in ([2, 4, 3], [2, 3, 4], [4, 2, 3], [1, 3, 2, 2])]

# Bài 2: duyệt 20, 11, 6, 4, 3, 12 (pop 3,4,6,11 -> vk = 11), rồi x = 6 < 11 -> dừng ở bước 7
nums2 = [6, 12, 3, 4, 6, 11, 20]
vk = float("-inf"); stk = []; step = 0
for x in reversed(nums2):
    step += 1
    if x < vk:
        break
    while stk and stk[-1] < x:
        vk = stk.pop()
    stk.append(x)
ans2_step = 7
ans2_triple = (6, 12, 11)        # 6 < 11 < 12, vị trí 0 < 1 < 5
assert (step, x, vk) == (ans2_step, 6, 11)
i, j, k = (nums2.index(v) for v in ans2_triple)
assert i < j < k and ans2_triple[0] < ans2_triple[2] < ans2_triple[1]
print("OK bài 1-2: dừng ở bước", step, "x =", x, "vk =", vk, "bộ ba", ans2_triple)

OK bài 1-2: dừng ở bước 7 x = 6 vk = 11 bộ ba (6, 12, 11)


In [35]:
# Bài 3
def find132_todo(nums):
    vk = float("-inf")
    stk = []
    for x in reversed(nums):
        if x < vk:
            return True
        while stk and stk[-1] < x:
            vk = stk.pop()
        stk.append(x)
    return False

# Bài 4
def find132_oracle(nums):
    return any(a < c < b for a, b, c in combinations(nums, 3))

random.seed(19)
for _ in range(300):
    nums = [random.randint(-5, 5) for _ in range(random.randint(1, 8))]
    assert find132_todo(nums) == find132_oracle(nums) == Solution().find132pattern(nums)
print("OK bài 3-4")

OK bài 3-4


In [36]:
# Bài 5(a): mẫu 231 (nums[k] < nums[i] < nums[j]); duyệt trái->phải, ngăn xếp giảm, vi = "2" lớn nhất bị pop...
# đối xứng bằng cách đảo dấu và đảo thứ tự: 231 trên a  <=>  132 trên (-a đảo ngược)? Kiểm chứng trực tiếp bằng oracle.
def has_231(nums):
    vi = float("-inf")     # giá trị lớn nhất của "2" (= nums[i]) đã có "3" đứng sau nó
    stk = []
    for x in nums:
        if x < vi:
            return True
        while stk and stk[-1] < x:
            vi = stk.pop()
        stk.append(x)
    return False

def oracle_231(nums):
    return any(c < a < b for a, b, c in combinations(nums, 3))

random.seed(51)
for _ in range(500):
    nums = [random.randint(-4, 4) for _ in range(random.randint(1, 8))]
    assert has_231(nums) == oracle_231(nums), nums

# Bài 5(b): đếm số bộ ba 132 trong O(n^2): với mỗi cặp (j, k), j<k, nums[j]>nums[k], đếm i<j có nums[i]<nums[k]
def count_132_fast(nums):
    n = len(nums)
    total = 0
    for k in range(n):
        cnt_i = 0                      # số i < j với nums[i] < nums[k]; cập nhật khi j tăng
        for j in range(k):
            if nums[j] > nums[k]:
                total += cnt_i
            if nums[j] < nums[k]:
                cnt_i += 1
    return total

for _ in range(300):
    nums = [random.randint(-4, 4) for _ in range(random.randint(1, 9))]
    assert count_132_fast(nums) == count_132(nums), nums
print("OK bài 5")

OK bài 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ngăn xếp đơn điệu duyệt từ phải sang trái, nhớ "2" lớn nhất)

```python
class Solution:
    def find132pattern(self, nums: List[int]) -> bool:
        vk = float("-inf")           # giá trị "2" lớn nhất đã chắc chắn có "3" đứng trước
        stk = []                     # ứng viên "3", giảm dần từ đáy lên đỉnh
        for x in reversed(nums):
            if x < vk:               # x là "1"
                return True
            while stk and stk[-1] < x:
                vk = stk.pop()       # phần tử bị x chặn trở thành "2"
            stk.append(x)
        return False
```

**Checklist**

- [ ] Hiểu thứ tự **1, 3, 2** (nhỏ nhất đứng trước, lớn nhất ở giữa, vừa ở cuối), điều kiện **nghiêm ngặt**?
- [ ] `vk` khởi tạo bằng `-inf` (không phải `0`), vì có số âm?
- [ ] Pop với `stk[-1] < x` (không phải `<=`)?
- [ ] Kiểm tra `x < vk` **trước** khi pop/push?
- [ ] Đã thử `[1,2,3,4]` (False), `[3,1,4,2]` (True), `[-1,3,2,0]` (True), `[1,2,2]` (False), `[1,0,1,-4,-3]` (False)?
- [ ] Skeleton đúng `class Solution` / `def find132pattern(self, nums: List[int]) -> bool`?